# Keras による高度な特徴量エンジニアリング

**学習目標**
1. 状態を持つ前処理レイヤ（stateful preprocessing layers）の定義と適応（adapt）を行う。
2. `Lambda` レイヤを使用してカスタム変換を適用する。

## 概要
このノートブックでは、Keras を使用してタクシー運賃予測モデルを構築し、特徴量エンジニアリングを活用してニューヨーク市のタクシー乗車運賃予測の精度を向上させます。

## セットアップ
このラボに必要なライブラリをインポートすることから始めます。

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import datetime
import shutil

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Model
from keras.callbacks import TensorBoard
from keras.layers import (
    CategoryEncoding,
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
    Lambda,
)
from keras.metrics import RootMeanSquaredError
from matplotlib import pyplot as plt

In [ ]:
%matplotlib inline

### 生データの読み込み

一連の最初のノートブックで作成した CSV ファイルを使用して、taxifare データセットを利用します。これらのファイルは `../data` に保存されています。

In [ ]:
!ls -l ../data/taxi-*.csv

### tf.data を使用した CSV ファイルの読み込み

上記の CSV ファイルからデータを読み込むためのこれらの関数は、[前のノートブック](2_dataset_api.ipynb)で作成しました。

`tf.data` API は効率的にデータを読み込み、前処理を行います。
- `parse_csv`: CSV の行を特徴量とラベルに解析します。複数の入力に対する Functional API との互換性のため、特徴量はタプルとして返されます。
- `create_dataset`: CSV ファイルから `tf.data.Dataset` を構築します。これには、`parse_csv` のマッピング、リピート、シャッフル（訓練用）、バッチ処理が含まれます。

In [ ]:
def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(ds[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # 一部の特徴量のみを使用
    # 個別に処理できるようにタプルで特徴量を渡す
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, mode="eval"):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(parse_csv).repeat()
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

## 高度な特徴量エンジニアリングの定義
次に、特徴量エンジニアリングをさらに追加してパフォーマンスの向上を試みます。
1.  **正規化（Normalization）:** 距離の計算やその他の処理を行う前に座標に適用します。
2.  **ユークリッド距離（Euclidean Distance）:** `Lambda` レイヤを使用して計算します。


### `Normalization` レイヤによる特徴量正規化の設定

`keras.layers.Normalization` レイヤは、平均を 0、分散を 1 にスケーリングすることで特徴量を標準化します。

このレイヤには状態（平均と分散）が必要となるため、次のいずれかを行う必要があります。
1. 状態の値をあらかじめ計算し、それを使ってレイヤをインスタンス化する。
```python
keras.layers.Normalization(mean=..., variance=...)
```
2. または、`adapt()` メソッドを使用して値を計算する。

ここでは後者のオプションを見てみましょう。

まず、これらの統計量を計算するためにデータを読み込みます。ここでは緯度と経度の列を取得します。

In [ ]:
def parse_lat_lon(row):
    columns = tf.strings.split(row, ",")
    # 緯度のインデックス: 3 および 5、経度のインデックス: 2 および 4
    lat_strings = tf.gather(columns, [3, 5])
    lon_strings = tf.gather(columns, [2, 4])
    lat_features = tf.strings.to_number(lat_strings)
    lon_features = tf.strings.to_number(lon_strings)
    return lat_features, lon_features


# ファイルからデータセットを作成
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)
lat_lon_ds = ds.map(parse_lat_lon).batch(10000)

次に、`Normalization` レイヤ（`lat_scaler`、`lon_scaler`）をインスタンス化し、読み込んだ緯度と経度の値に対してそれぞれの `adapt()` メソッドを使用して平均と分散を学習します。
適合（adapt）されたこれらのレイヤは、モデル内で学習済みの正規化を適用するために使用できます。

In [ ]:
lat_scaler = keras.layers.Normalization(axis=None)
lon_scaler = keras.layers.Normalization(axis=None)

lat_values, lon_values = iter(lat_lon_ds).__next__()

lat_scaler.adapt(lat_values)
lon_scaler.adapt(lon_values)

print("Computed statistics for latitude:")
print(f"mean: {lat_scaler.mean}, variance: {lat_scaler.variance}")
print(f"+++++")
print("Computed statistics for longitude:")
print(f"mean: {lon_scaler.mean}, variance: {lon_scaler.variance}")

### 入力層の定義

In [ ]:
INPUT_COLS = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]

# 入力層はすべて float 型
inputs = {
    colname: Input(name=colname, shape=(1,), dtype="float32")
    for colname in INPUT_COLS
}

### カスタム特徴量: `Lambda` レイヤによるユークリッド距離

`euclidean` 関数は直線距離を計算します。後ほど [`keras.layers.Lambda` レイヤ](https://keras.io/api/layers/core_layers/lambda/) を使用してこの関数をラップし、特徴量エンジニアリングのために Keras モデルに直接統合できるようにします。これにより、前処理をモデルと一体化して保持できます。

In [ ]:
def euclidean(params):
    lon1, lat1, lon2, lat2 = params
    londiff = lon2 - lon1
    latdiff = lat2 - lat1
    return keras.ops.sqrt(londiff * londiff + latdiff * latdiff)

### 前処理、正規化、Lambda レイヤ統合の定義

特徴量エンジニアリングの手順を適用します。
1. **バケット境界:** 正規化されたデータに合わせて調整（範囲 `[-5, 5]`）。
2. **正規化:** 生の座標は、適合済みの `lon_scaler` および `lat_scaler` を使用してスケーリングされます。
3. **Lambda レイヤ:** `euclidean` 関数が、これらの*正規化された*座標に基づいて距離を計算します。
4. **離散化（Discretization）:** 正規化された座標をバケット化します。
5. **特徴量のクロスと埋め込み（Feature Crossing & Embedding）:** （正規化および離散化された）特徴量に適用されます。
6. **連結（Concatenate）:** `euclidean_distance` と最終的な `pd_embed` を結合して DNN に渡します。

**演習**: Lambda レイヤと前処理レイヤを完成させて、特徴量エンジニアリングのロジックを定義してください。

In [ ]:
NBUCKETS = 16

latbuckets = np.linspace(start=-5, stop=5, num=NBUCKETS).tolist()
lonbuckets = np.linspace(start=-5, stop=5, num=NBUCKETS).tolist()

# 経度を正規化
scaled_plon = lon_scaler(inputs["pickup_longitude"])
scaled_dlon = ...  # TODO

# 緯度を正規化
scaled_plat = ...  # TODO
scaled_dlat = ...  # TODO

# カスタム euclidean 関数のための Lambda レイヤ
euclidean_distance = Lambda(...)

# 離散化（Discretization）
plon = Discretization(lonbuckets, name="plon_bkt")(scaled_plon)
plat = ...  # TODO
dlon = ...  # TODO
dlat = ...  # TODO


# HashedCrossing レイヤによる特徴量のクロス
p_fc = HashedCrossing(num_bins=(NBUCKETS + 1) ** 2, name="p_fc")((plon, plat))
d_fc = ...  # TODO
pd_fc = ...  # TODO

# Embedding レイヤによる埋め込み
pd_embed = Flatten()(
    Embedding(input_dim=(NBUCKETS + 1) ** 4, output_dim=10, name="pd_embed")(
        pd_fc
    )
)

deep = Concatenate()(
    [
        scaled_plon,
        scaled_dlon,
        scaled_plat,
        scaled_dlat,
        euclidean_distance,
        pd_embed,
    ]
)

### DNN レイヤの定義

連結された `euclidean_distance` と `pd_embed` のテンソルは、`Dense` レイヤに渡されます。

In [ ]:
dnn_hidden_units = [32, 8]

# 隠れ Dense レイヤを追加
for i, num_nodes in enumerate(dnn_hidden_units, start=1):
    deep = Dense(num_nodes, activation="relu", name=f"hidden_{i}")(deep)

# 回帰問題のため最終出力は線形活性化
output = Dense(1, activation="linear", name="fare")(deep)

### エンジニアリングされたモデルのインスタンス化とコンパイル

元の入力と最終的にエンジニアリングされた出力を使用して Keras モデルを定義します。

In [ ]:
model = keras.Model(inputs=list(inputs.values()), outputs=output)

# モデルをコンパイル
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[RootMeanSquaredError()],
)

モデルのアーキテクチャがどのように変化したか確認してみましょう。

In [ ]:
keras.utils.plot_model(model, show_layer_names=True, rankdir="LR")

## エンジニアリングされたモデルの訓練

以前と同じ設定を使用して、新しいモデルを訓練します。

In [ ]:
BATCH_SIZE = 64
NUM_TRAIN_EXAMPLES = 10000 * 10  # 訓練データセットはリピート（ラップアラウンド）される
NUM_EVALS = 10  # 評価を行う回数
NUM_EVAL_EXAMPLES = 1000

In [ ]:
trainds = create_dataset(
    pattern="../data/taxi-train.csv", batch_size=BATCH_SIZE, mode="train"
)

evalds = create_dataset(
    pattern="../data/taxi-valid.csv", batch_size=BATCH_SIZE, mode="eval"
).take(NUM_EVAL_EXAMPLES // BATCH_SIZE)

steps_per_epoch = NUM_TRAIN_EXAMPLES // (BATCH_SIZE * NUM_EVALS)

history = model.fit()  # TODO

前と同様に、DNN モデルのレイヤを可視化してみましょう。

In [ ]:
RMSE_COLS = ["root_mean_squared_error", "val_root_mean_squared_error"]

pd.DataFrame(history.history)[RMSE_COLS].plot()

上記で扱った例に対して、特徴量エンジニアリングを施したこの新しいモデルを使用して予測を行ってみましょう。

In [ ]:
model.predict(
    {
        "pickup_longitude": keras.ops.convert_to_tensor([-73.982683]),
        "pickup_latitude": keras.ops.convert_to_tensor([40.742104]),
        "dropoff_longitude": keras.ops.convert_to_tensor([-73.983766]),
        "dropoff_latitude": keras.ops.convert_to_tensor([40.755174]),
        "passenger_count": keras.ops.convert_to_tensor([3.0]),
    },
    steps=1,
)

Copyright 2025 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.